# Detector de Enfermedades en Hojas — Experimentos CNN
**Módulo:** CNN + Evaluación + Comparativa (Henry Huanca)

Este notebook documenta los experimentos del segundo método (CNN con transfer learning), las curvas de entrenamiento y la comparativa final contra el pipeline clásico de Jose.

> Ejecutar desde la carpeta `notebooks/`. Las celdas que dependen del dataset están protegidas con `try/except` para que el notebook no se rompa si todavía no están las imágenes en `data/splits/`.

## 1. Configuración del entorno

In [ ]:
import sys, os
# Agregar backend/ al path para importar los módulos
sys.path.insert(0, os.path.abspath('../backend'))
os.chdir('../backend')   # los módulos asumen el cwd en backend/

import numpy as np
import matplotlib.pyplot as plt

from utils import load_config, get_device, get_section, count_parameters
from cnn_model import build_model, DEFAULT_CNN

config = load_config()
device = get_device()
print('Dispositivo:', device)

## 2. Arquitectura del modelo (transfer learning)
Backbone preentrenado en ImageNet con la cabeza reemplazada por un clasificador de 2 clases (sana / enferma).

In [ ]:
ccfg = get_section(config, 'cnn', DEFAULT_CNN)
model = build_model(
    backbone=ccfg['backbone'],
    num_classes=len(ccfg['class_names']),
    pretrained=ccfg['pretrained'],
    freeze_backbone=ccfg['freeze_backbone'],
    dropout=ccfg['dropout'],
)
total, trainable = count_parameters(model)
print(f"Backbone: {ccfg['backbone']}")
print(f'Parámetros totales:    {total:,}')
print(f'Parámetros entrenables:{trainable:,}')

## 3. Datos y data augmentation
Se cargan los splits de `data/splits/` con la estructura ImageFolder. El conjunto de entrenamiento aplica recortes, giros, rotación y variación de color.

In [ ]:
from train import build_dataloaders, DEFAULT_TRAINING
tcfg = get_section(config, 'training', DEFAULT_TRAINING)
try:
    loaders, class_names, counts = build_dataloaders(config, tcfg, ccfg)
    print('Clases:', class_names)
    print('Imágenes de entrenamiento por clase:', counts.tolist())
except FileNotFoundError as e:
    print('Dataset aún no disponible:', e)

## 4. Entrenamiento
Ejecuta el entrenamiento (o reduce `--epochs` para una prueba rápida). El mejor modelo se guarda en `models/cnn_weights.pth`.

In [ ]:
from train import train, plot_history
# Para una corrida real: model, meta, history = train(config)
# Para una prueba rápida de 3 épocas:
try:
    model, meta, history = train(config, overrides={'epochs': 3})
    plot_history(history)
    plt.show()
except FileNotFoundError as e:
    print('Necesitas el dataset en data/splits/ para entrenar:', e)

## 5. Evaluación de la CNN
Métricas de clasificación y matriz de confusión sobre el conjunto de test.

In [ ]:
from cnn_model import load_model
from evaluate import evaluate_cnn, plot_confusion_matrix
try:
    model, meta = load_model('models/cnn_weights.pth', config, device)
    metrics = evaluate_cnn(model, loaders['test'], device, meta['class_names'])
    print('Accuracy:', metrics['accuracy'], '| Macro F1:', metrics['macro_f1'])
    plot_confusion_matrix(metrics['confusion_matrix'], meta['class_names'],
                          title='CNN — Matriz de confusión')
    plt.show()
except FileNotFoundError as e:
    print('Entrena primero el modelo:', e)

## 6. Comparativa: clásico vs CNN
Núcleo del aporte original. Ejecuta ambos pipelines sobre el mismo test y genera `outputs/metrics/comparison.json` para el frontend.

In [ ]:
from comparison import run_comparison, print_comparison_table
from utils import save_json
try:
    result = run_comparison(config, weights_path='models/cnn_weights.pth')
    print_comparison_table(result)
    save_json(result, 'outputs/metrics/comparison.json')
except Exception as e:
    print('No se pudo correr la comparativa todavía:', e)

## 7. Conclusiones
- **Velocidad:** comparar `avg_ms` de cada método.
- **Precisión:** comparar accuracy y F1.
- **Tipo de errores:** revisar `errors` (en qué imágenes falla cada uno).
- **Localización:** el `mean_iou` indica cuánto coincide la zona enferma del método clásico con la atención de la CNN (Grad-CAM).

_Completar con los números reales una vez entrenado el modelo._